# 05 — Frozen 2024 evaluation, full Task 1 and reload evidence
This notebook reads the immutable model manifest. Do not change configuration/models after inspecting these test-year outputs.

Prerequisites: notebook 00 has uploaded the project files and inputs for the selected run; notebook 01 has completed the cleaned and coverage Parquet outputs. This notebook loads the config from the project checkout and reads those existing outputs directly. This notebook has not yet been executed.

Set `MODEL_ATTEMPT` to the same unused attempt ID in Notebooks 04 and 05. The notebook checks the frozen model manifest and model root before reading final-test rows, then refuses to overwrite an existing final-evaluation prefix.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import SparkSession

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path[:0] = [str(PROJECT_ROOT / "src"), str(PROJECT_ROOT)]

from rainfall.config import load_config, config_hash
from rainfall.runtime import spark_action

BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261012-full-01")
MODEL_ATTEMPT = os.environ.get("MODEL_ATTEMPT", "attempt-01")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
config = load_config(PROJECT_ROOT / "config" / "experiment.json")
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")


def cloud_read(args):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Cloud command failed.")
    return result.stdout

In [ ]:
# Install the pinned plotting wheel into the project vendor directory if needed.
# This private, cluster-local path is visible to the active Jupyter kernel.
vendor = PROJECT_ROOT / "vendor"
scienceplots_package = vendor / "scienceplots"
if not scienceplots_package.is_dir():
    wheel = PROJECT_ROOT / "wheels" / "scienceplots-2.2.2-py3-none-any.whl"
    if not wheel.is_file():
        raise RuntimeError("The pinned SciencePlots wheel is missing from the verified project bundle.")
    vendor.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--target", str(vendor), str(wheel)], check=True)
sys.path.insert(0, str(vendor))
import scienceplots
print("SciencePlots report styles registered from the project vendor directory.")

In [ ]:
from rainfall.experiments import final_evaluation, read_record
from rainfall.plotting import evaluation_figures
MODEL_ROOT = f"{ROOT}/models/{MODEL_ATTEMPT}"
FROZEN_MANIFEST = f"{MODEL_ROOT}/frozen_manifest"
with spark_action(spark, RUN_ID, "read_frozen_manifest"):
    frozen = read_record(spark, FROZEN_MANIFEST)
if frozen["config_sha256"] != config_hash(config) or frozen["model_root"] != MODEL_ROOT:
    raise RuntimeError("The selected model attempt or configuration differs from the frozen manifest.")
EVALUATION_ROOT = f"{ROOT}/final/{MODEL_ATTEMPT}"
evaluation_path = spark._jvm.org.apache.hadoop.fs.Path(EVALUATION_ROOT)
evaluation_fs = evaluation_path.getFileSystem(spark._jsc.hadoopConfiguration())
if evaluation_fs.exists(evaluation_path):
    raise FileExistsError(
        f"{EVALUATION_ROOT} already contains evaluation artifacts; do not rerun or overwrite final-test outputs."
    )
with spark_action(spark, RUN_ID, "final_test_evaluation"):
    base = spark.read.parquet(f"{ROOT}/base_features")
    evaluation = final_evaluation(base, frozen, EVALUATION_ROOT)
    display(evaluation)
    # First three supported thresholds, fixed from training before test.
    names = ["climatology", "lr_B", "gbt_B"]
    curves = {name: spark.read.json(f"{EVALUATION_ROOT}/pr_curves/{name}") for name in names}
    reliability_tables = {name: spark.read.json(f"{EVALUATION_ROOT}/reliability/{name}") for name in names}
    evaluation_figures(curves, reliability_tables, PROJECT_ROOT / "figures" / "final")

In [ ]:
from rainfall.eda import explore
from rainfall.plotting import pattern_figures, eda_figures, export_eda_tables
if frozen["config_sha256"] != config_hash(config):
    raise RuntimeError("Full-period EDA requires the frozen Task 2 configuration.")
with spark_action(spark, RUN_ID, "task1_full_period_eda"):
    clean = spark.read.parquet(f"{ROOT}/cleaned") 
    clean = clean.persist(StorageLevel.DISK_ONLY)
    months = spark.read.parquet(f"{ROOT}/coverage_month")
    years = spark.read.parquet(f"{ROOT}/coverage_year")
    try:
        aggregates = explore(clean, months, years, config, include_test=True)
        # Materialise once. Plotting/export reads these small tables, not raw rows.
        materialised = {}
        for name, frame in aggregates.items():
            destination = f"{ROOT}/task1/{name}"
            frame.write.mode("errorifexists").json(destination)
            materialised[name] = spark.read.schema(frame.schema).json(destination)
            print(f"\n{name} — full table saved to {destination}")
            materialised[name].show(20, truncate=False)
        figure_dir = PROJECT_ROOT / "figures" / "final"
        pattern_figures(materialised, years, config, figure_dir)
        eda_figures(materialised, figure_dir)
        findings = export_eda_tables(materialised, PROJECT_ROOT / "eda_tables" / "final")
        display(findings)
    finally:
        clean.unpersist()
cloud_read(["storage", "rsync", "--recursive", str(PROJECT_ROOT / "eda_tables" / "final"), f"{ROOT}/task1/report_tables"])
cloud_read(["storage", "rsync", "--recursive", str(figure_dir), f"{ROOT}/task1/figures"])


### Full-period EDA interpretation

The 2024 tables above are descriptive outputs obtained after the configuration/model freeze. Use `task1/report_tables/findings.json` together with coverage, cohort sizes and exported CSVs to write quantified within-year and across-year findings. Counts and tail summaries are observational; no causal or long-term climate claims follow from these eight years. Jobs summary and every Spark Job detail still require actual Spark UI screenshots.

In [ ]:
# Download the selected deployment package to a portable local directory.
from rainfall.config import write_json
from rainfall.predict import RainfallPredictor
package = PROJECT_ROOT / "models" / "deployment"
package.mkdir(parents=True, exist_ok=True)
cloud_read(["storage", "rsync", "--recursive", frozen["selected"]["path"], str(package / "selected")])
cloud_read(["storage", "rsync", "--recursive", f"{frozen['model_root']}/training_stations", str(package / "training_stations")])
if frozen["use_calibration"]:
    cloud_read(["storage", "rsync", "--recursive", f"{frozen['model_root']}/calibrator", str(package / "calibrator")])
write_json(package / "metadata.json", {"algorithm": frozen["algorithm"], "config": config, "use_calibration": frozen["use_calibration"], "selected": frozen["selected"], "manifest_sha256": frozen["manifest_sha256"]})
# A synthetic history fixture enables independent load/inference without raw data.
fixture_epoch = 1704081600
fixture_rows = [("S24", fixture_epoch - i * 300, 0.0, 0) for i in range(36)]
history = spark.createDataFrame(fixture_rows, "station_id string, epoch long, rain_mm double, rain_units long")
history.write.mode("errorifexists").parquet(str(package / "history_fixture"))
with spark_action(spark, RUN_ID, "fresh_model_reload_and_inference"):
    loaded = RainfallPredictor.load(spark, str(package), spark.read.parquet(str(package / "history_fixture")))
    from datetime import datetime, timezone
    query_time = datetime.fromtimestamp(fixture_epoch, timezone.utc).isoformat()
    display(loaded.predict("S24", query_time, 1.0))
    assert loaded.predict("S24", query_time, 20.0)["status"] == "threshold_out_of_domain"
    assert loaded.predict("S24", "2024-01-01T12:00:01+08:00", 1.0)["status"] == "off_grid_time"
# Upload portable model and report figures; no raw data is packaged.
cloud_read(["storage", "rsync", "--recursive", str(package), f"{ROOT}/deployment"])
cloud_read(["storage", "rsync", "--recursive", str(PROJECT_ROOT / "figures" / "final"), f"{ROOT}/figures"])
print("Portable deployment package and report figures saved.")

In [ ]:
spark.stop()